In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"

OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# LOAD DATA
df = pd.read_csv(INPUT_CSV)
print("="*70)
print(" GBM THICKNESS GMM VISUALIZATION ")
print("="*70)

# ============================================================
# ANALYSE EACH PATIENT
# ============================================================

for patient_id, patient_data in df.groupby("patient_id"):

    # Thickness values
    thickness = (
        patient_data["median_thickness_nm"]
        .dropna()
        .values
    )

    if len(thickness) < 5:
        print(
            f"{patient_id}: Not enough samples"
        )
        continue

    X = thickness.reshape(-1,1)

    # --------------------------------------------------------
    # Compare 1,2,3 Gaussian models
    # --------------------------------------------------------

    models = {}
    bic_values = {}

    for n_components in [1,2,3]:
        gmm = GaussianMixture(

            n_components=n_components,
            random_state=42,
            n_init=20
        )

        gmm.fit(X)
        models[n_components] = gmm
        bic_values[n_components] = gmm.bic(X)

    # Select best model
    best_components = min(
        bic_values,
        key=bic_values.get
    )
    best_gmm = models[best_components]

    print(
        f"{patient_id}: Best model = {best_components} peaks"
    )

    # ========================================================
    # PLOT DISTRIBUTION
    # ========================================================

    plt.figure(figsize=(9,5))

    plt.hist(
        thickness,
        bins=15,
        density=True,
        alpha=0.5,
        label="Observed thickness"
    )

    # Smooth x axis
    x = np.linspace(
        thickness.min()-50,
        thickness.max()+50,
        1000
    )
    x_plot = x.reshape(-1,1)

    # Total GMM distribution
    log_probability = best_gmm.score_samples(x_plot)
    probability = np.exp(log_probability)

    plt.plot(
        x,
        probability,
        linewidth=2,
        label=f"GMM ({best_components} components)"
    )

    # Individual Gaussian components

    for i in range(best_components):
        mean = best_gmm.means_[i][0]
        std = np.sqrt(
            best_gmm.covariances_[i][0][0]
        )
        weight = best_gmm.weights_[i]

        gaussian = (
            weight *
            (1/(std*np.sqrt(2*np.pi))) *

            np.exp(
                -0.5*((x-mean)/std)**2
            )
        )

        plt.plot(
            x,
            gaussian,
            linestyle="--",
            label=f"Peak {i+1}: {mean:.1f} nm"
        )

    
    plt.title(

        f"GBM Thickness Distribution - Patient {patient_id}"
    )

    plt.xlabel(
        "Median GBM thickness (nm)"
    )

    plt.ylabel(
        "Density"
    )

    plt.legend()
    plt.tight_layout()

    save_path = os.path.join(
        OUTPUT_FOLDER,
        f"GMM_fit_{patient_id}.png"
    )

    plt.savefig(
        save_path,
        dpi=300
    )
    plt.close()

print("\n")
print("="*70)
print("ALL GMM PLOTS GENERATED")
print("="*70)
print(
    f"Saved in: {OUTPUT_FOLDER}"
)

 GBM THICKNESS GMM VISUALIZATION 
01-24: Best model = 3 peaks
02-24: Best model = 3 peaks
03-24: Best model = 2 peaks
04-23: Best model = 3 peaks
05-24: Best model = 3 peaks
06-24: Best model = 2 peaks
07-25: Best model = 3 peaks
08-25: Best model = 3 peaks
09-24: Best model = 2 peaks
10-24: Best model = 2 peaks
11-24: Best model = 2 peaks


ALL GMM PLOTS GENERATED
Saved in: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots
